In [ ]:
%matplotlib widget

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Box, Layout, Output, Label
from IPython.display import display, Latex, Markdown, Image, Math

import params as st
from model import nonlinSys, stationarySys
from feedforward import feedForwardIO, feedForwardFlat, feedBackIO, feedBackFlat

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
def make_box_layout():
     return widgets.Layout(
        border='solid 1px black',
        margin='0px 5px 5px 0px',
        padding='2px 2px 2px 2px'
     )

# Regelung des Zweitanksystems: Eingangs-Ausgangslinearisierung und flacher Ausgang

In [ ]:
imag = Image("../images/twoTank.png", width=450)

outL = Output()
outR = Output()

with outL:
    display(imag)
with outR:
    display(Markdown(r"""
**Modellgleichungen**

Zufluss $u(t) \geq 0$ der Pumpe in Tank 1, Ausfluss nach Torricelli $q_i(h_i) = a_i \sqrt{2 g h_i}$
\begin{align*}
    A_1 \dot{h}_1(t) & = u(t) - q_1(h_1(t)) \\
    A_2 \dot{h}_2(t) & = q_1(h_1(t)) - q_2(h_2(t))
\end{align*}

**Eingangs-Ausgangslinearisierung mit $y = h_1$** ($r = 1$)
\begin{align*}
    u(t) & = q_1(h_1(t)) + A_1 v(t), &
    v(t) & = \dot{y}_\mathrm{r}(t) + s_\mathrm{R} \left(y(t) - y_\mathrm{r}(t)\right)
\end{align*}
Fehlerdynamik $\dot{e} = s_\mathrm{R} e$, $h_2$ folgt aus der stabilen internen Dynamik.

**Flachheitsbasierte Regelung mit $y = h_2$** ($r = n = 2$)
\begin{align*}
    u(t) & = q_1(h_1) + \frac{A_1}{q_1'(h_1)} \left(A_2 v(t) + q_2'(h_2) \dot{y}\right), &
    \dot{y} & = \frac{q_1(h_1) - q_2(h_2)}{A_2} \\
    v(t) & = \ddot{y}_\mathrm{r}(t) + 2 s_\mathrm{R} \left(\dot{y} - \dot{y}_\mathrm{r}(t)\right) - s_\mathrm{R}^2 \left(y - y_\mathrm{r}(t)\right)
\end{align*}
Fehlerdynamik $\ddot{e} - 2 s_\mathrm{R} \dot{e} + s_\mathrm{R}^2 e = 0$ mit doppeltem Eigenwert $s_\mathrm{R}$, keine interne Dynamik.

**Optionaler I-Anteil** mit $z(t) = \int_0^t e(\tau) \,\mathrm{d}\tau$ und $e = y - y_\mathrm{r}$

$y = h_1$ |   | $y = h_2$
----------------------|----|------------------------
\begin{align*}
    v & = \dot{y}_\mathrm{r} + 2 s_\mathrm{R} e - s_\mathrm{R}^2 z
\end{align*} |   | \begin{align*}
    v & = \ddot{y}_\mathrm{r} + 3 s_\mathrm{R} \dot{e} - 3 s_\mathrm{R}^2 e + s_\mathrm{R}^3 z
\end{align*}

Während die Pumpe in der Begrenzung $u \geq 0$ ist, wird der Integrator angehalten (Anti-Windup).
    """))
cols = HBox([outL, outR], layout=Layout(display='flex', flex_flow='row', justify_content='space-around', align_items='center'))
display(cols)

**Definition Parameter**

In [ ]:
paramsNom = st.g, st.A1, st.A2, st.a1, st.a2
tSim = np.linspace(0, 400, 801)
t0 = 20
h10 = 0.2

In [ ]:
errorIO = lambda t, x, ffArgs: x[0] - feedForwardIO(t, *ffArgs)[0]
errorFlat = lambda t, x, ffArgs: x[1] - feedForwardFlat(t, *ffArgs)[0][1]

def simulate(feedBack, error, sR, integral, ffArgs, x0, params):
    def sys(t, x):
        uC = feedBack(t, x, sR, ffArgs, integral)
        dx = np.zeros(3)
        dx[0:2] = nonlinSys(t, x[0:2], lambda _t: uC, params)
        # Integrator anhalten, solange die Pumpe in der Begrenzung ist
        if integral and uC >= 0:
            dx[2] = error(t, x, ffArgs)
        return dx

    res = solve_ivp(sys, [tSim[0], tSim[-1]], [*x0, 0], t_eval=tSim, max_step=1)
    uAll = np.array([feedBack(t, res.y[:, idx], sR, ffArgs, integral) for idx, t in enumerate(res.t)])
    return res, uAll

## Simulation

In [ ]:
output = widgets.Output()

with output:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.subplots_adjust(bottom=0.08, top=0.9, left=0.1, right=0.95, hspace=0.1)

ax1.set_xlim([tSim[0], tSim[-1]])
for ax in [ax1, ax2, ax3]:
    ax.grid()
ax1.set_ylabel(r"$u$ in l/s")
ax2.set_ylabel(r"$h_1$ in m")
ax3.set_ylabel(r"$h_2$ in m")
ax3.set_xlabel(r"$t$ in s")

def setLimits(ax, *data):
    data = np.concatenate([np.atleast_1d(d) for d in data])
    dMin, dMax = np.min(data), np.max(data)
    ax.set_ylim(dMin - 0.1 * np.abs(dMax - dMin) - 1e-3, dMax + 0.1 * np.abs(dMax - dMin) + 1e-3)

lineUIO, = ax1.plot([], [], 'C0', label=r'$y = h_1$')
lineUIOd, = ax1.plot([], [], 'C0:')
lineUFlat, = ax1.plot([], [], 'C1', label=r'$y = h_2$')
lineUFlatd, = ax1.plot([], [], 'C1:')
lineUd, = ax1.plot([], [], 'k:', label=r'$u$ ohne Begrenzung')
lineH1IO, = ax2.plot([], [], 'C0')
lineH1Flat, = ax2.plot([], [], 'C1')
lineH1Ref, = ax2.plot([], [], 'k--', label=r'Referenz')
lineH2IO, = ax3.plot([], [], 'C0')
lineH2Flat, = ax3.plot([], [], 'C1')
lineH2Ref, = ax3.plot([], [], 'k--')

handlesAx, labelsAx = [], []
for ax in [ax1, ax2, ax3]:
    h, la = ax.get_legend_handles_labels()
    handlesAx += h
    labelsAx += la

fig.legend(handlesAx, labelsAx,
           bbox_to_anchor=(0.1, 0.91, 0.85, .15), loc=3,
           ncol=4, mode="expand", borderaxespad=0., framealpha=0.5)

sliderDh = widgets.FloatSlider(value=0.1,
                               min=-0.15,
                               max=0.15,
                               step=0.01,
                               description=r"$\Delta h_1$")
sliderT = widgets.FloatSlider(value=100,
                              min=30,
                              max=300,
                              step=10,
                              description=r"$T$")
sliderPoleK = widgets.FloatSlider(value=-0.05,
                                  min=-0.3,
                                  max=-0.01,
                                  step=0.01,
                                  description=r"$s_{\text{R}}$")
sliderDh10 = widgets.FloatSlider(value=0,
                                 min=-0.1,
                                 max=0.1,
                                 step=0.01,
                                 description=r"$\Delta h_1(0)$")
sliderDh20 = widgets.FloatSlider(value=0,
                                 min=-0.1,
                                 max=0.1,
                                 step=0.01,
                                 description=r"$\Delta h_2(0)$")
sliderDa = widgets.FloatSlider(value=0,
                               min=-20,
                               max=20,
                               step=1,
                               description=r"$\Delta a_1$ in %")
checkIO = widgets.Checkbox(value=True,
                           description=r'$y = h_1$')
checkFlat = widgets.Checkbox(value=True,
                             description=r'$y = h_2$')
checkI = widgets.Checkbox(value=False,
                          description='I-Anteil')

def calcSystem(_):
    sR = sliderPoleK.value
    ffArgs = (h10, sliderDh.value, t0, sliderT.value)
    paramsSys = st.g, st.A1, st.A2, st.a1 * (1 + sliderDa.value / 100), st.a2
    x0 = [h10 + sliderDh10.value, stationarySys(h10, paramsNom)[1] + sliderDh20.value]

    h1Ref = np.array([feedForwardIO(t, *ffArgs)[0] for t in tSim])
    h2Ref = np.array([feedForwardFlat(t, *ffArgs)[0][1] for t in tSim])

    integral = checkI.value
    resIO, uIO = simulate(feedBackIO, errorIO, sR, integral, ffArgs, x0, paramsSys)
    resFlat, uFlat = simulate(feedBackFlat, errorFlat, sR, integral, ffArgs, x0, paramsSys)

    lines = [(checkIO.value, [(lineUIO, 1e3 * np.maximum(uIO, 0)), (lineUIOd, 1e3 * uIO),
                              (lineH1IO, resIO.y[0]), (lineH2IO, resIO.y[1])]),
             (checkFlat.value, [(lineUFlat, 1e3 * np.maximum(uFlat, 0)), (lineUFlatd, 1e3 * uFlat),
                                (lineH1Flat, resFlat.y[0]), (lineH2Flat, resFlat.y[1])])]
    for active, data in lines:
        for line, d in data:
            if active:
                line.set_data(tSim, d)
            else:
                line.set_data([], [])
    lineH1Ref.set_data(tSim, h1Ref)
    lineH2Ref.set_data(tSim, h2Ref)

    setLimits(ax1, 1e3 * uIO, 1e3 * uFlat)
    setLimits(ax2, h1Ref, resIO.y[0], resFlat.y[0])
    setLimits(ax3, h2Ref, resIO.y[1], resFlat.y[1])

    fig.canvas.draw()

for w in [sliderDh, sliderT, sliderPoleK, sliderDh10, sliderDh20, sliderDa, checkIO, checkFlat, checkI]:
    w.observe(calcSystem, names='value')

calcSystem(None)

bFeedForward = VBox([Label(value='Steuerung'), sliderDh, sliderT])
bFeedForward.layout = make_box_layout()
bOutput = VBox([Label(value='Ausgang'), checkIO, checkFlat])
bOutput.layout = make_box_layout()
bFeedBack = VBox([Label(value='Regler'), sliderPoleK, checkI])
bFeedBack.layout = make_box_layout()
bOutput.layout = make_box_layout()
bModel = VBox([Label(value='Strecke'), sliderDh10, sliderDh20, sliderDa])
bModel.layout = make_box_layout()
controls = HBox([bFeedForward, bOutput, bFeedBack, bModel])

VBox([controls, output], layout=Layout(display='flex', flex_flow='column', justify_content='center', align_items='center'))